# Inspect a recorded bench rollout

Loads the most recent `trajectory.h5` under `runs/`, prints its metadata + dataset layout, and plays the per-camera videos with `mediapy`.

In [ ]:
import tempfile
from pathlib import Path

import h5py
import mediapy as media

RUNS_DIR = Path("runs")

# Pick the most recently modified trajectory.h5; override TRAJ_PATH manually to inspect a specific one.
TRAJ_PATH = max(RUNS_DIR.rglob("trajectory.h5"), key=lambda p: p.stat().st_mtime)
print("Inspecting:", TRAJ_PATH)

## Metadata (HDF5 attrs)

In [ ]:
with h5py.File(TRAJ_PATH, "r") as f:
    for k in sorted(f.attrs):
        print(f"  {k} = {f.attrs[k]}")

## Dataset layout

In [ ]:
with h5py.File(TRAJ_PATH, "r") as f:
    def visit(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"  {name}/")
        else:
            print(f"  {name}  shape={obj.shape}  dtype={obj.dtype}")
    f.visititems(visit)

## Per-camera videos

Each camera stream is stored as a serialized MP4 byte blob under `observation/videos/{camera_id}`. We dump each blob to a temp file and play it with `mediapy`.

In [ ]:
with h5py.File(TRAJ_PATH, "r") as f:
    video_group = f["observation/videos"]
    cam_ids = list(video_group.keys())
    cam_bytes = {cam: bytes(video_group[cam][()]) for cam in cam_ids}

videos = {}
_tmp_files = []
for cam, data in cam_bytes.items():
    tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    tmp.write(data)
    tmp.flush()
    _tmp_files.append(tmp)
    videos[cam] = media.read_video(tmp.name)
    print(f"  {cam}: {videos[cam].shape}  ({len(data) / 1e6:.2f} MB)")

media.show_videos(videos, fps=15)